In [ ]:

import pandas as pd
from pathlib import Path

# Locate the CSV from the project root or python folder
file_path = Path("data/phase5_clu_modeling_dataset.csv")

if not file_path.exists():
    file_path = Path("../data/phase5_clu_modeling_dataset.csv")

# Load the modeling dataset
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


In [ ]:

# Normalize source names and historical labels for checking
source = df["dataset_source"].astype("string").str.strip().str.lower()
labels = df["went_on_backorder"].astype("string").str.strip().str.lower()

# Dataset checks
total_skus = df["sku"].nunique()
labelled_skus = (source.eq("train") & labels.isin(["true", "false"])).sum()
unlabelled_skus = (source.eq("test") & labels.isna()).sum()
backordered_skus = (source.eq("train") & labels.eq("true")).sum()
duplicate_skus = df["sku"].duplicated().sum()

print("Total unique SKUs:", total_skus)
print("Labelled SKUs:", labelled_skus)
print("Unlabelled SKUs:", unlabelled_skus)
print("Historical Backordered SKUs:", backordered_skus)
print("Duplicate SKUs:", duplicate_skus)

assert df.shape == (19053, 21), "Dataset shape mismatch"
assert total_skus == 19053, "SKU count mismatch"
assert labelled_skus == 15243, "Labelled count mismatch"
assert unlabelled_skus == 3810, "Unlabelled count mismatch"
assert backordered_skus == 1813, "Backorder count mismatch"
assert duplicate_skus == 0, "Duplicate SKUs found"

print("\nPhase 5 CSV Validation: PASS")


In [ ]:

# Phase 5 - Action 5.2.1A
# Inspect data types and missing values

print("DATASET INFORMATION")
print("-" * 50)

df.info()

print("\nCOLUMN DATA TYPES")
print("-" * 50)

print(df.dtypes.to_string())

print("\nMISSING VALUES")
print("-" * 50)

print(df.isna().sum().to_string())


In [ ]:

# Inspect target encoding and dataset source

print("BACKORDER TARGET VALUES")
print("-" * 50)

print(
    df["went_on_backorder"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nDATASET SOURCE VALUES")
print("-" * 50)

print(
    df["dataset_source"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nSOURCE VS TARGET")
print("-" * 50)

print(
    pd.crosstab(
        df["dataset_source"],
        df["went_on_backorder"],
        dropna=False
    ).to_string()
)


In [ ]:

# Phase 5 - Action 5.2.1B
# Separate labelled and unlabelled records

labelled_df = df[
    (df["dataset_source"] == "train") &
    (df["went_on_backorder"].notna())
].copy()

unlabelled_df = df[
    (df["dataset_source"] == "test") &
    (df["went_on_backorder"].isna())
].copy()

print("Labelled records:", len(labelled_df))
print("Unlabelled records:", len(unlabelled_df))
print("Total records:", len(labelled_df) + len(unlabelled_df))

assert len(labelled_df) == 15243
assert len(unlabelled_df) == 3810
assert len(labelled_df) + len(unlabelled_df) == len(df)

print("\nDataset separation: PASS")


In [ ]:

# Define approved preliminary predictors

feature_cols = [
    "national_inv",
    "in_transit_qty",
    "min_bank",
    "forecast_3_month",
    "forecast_6_month",
    "forecast_9_month",
    "sales_1_month",
    "sales_3_month",
    "sales_6_month",
    "sales_9_month",
    "lead_time",
    "perf_6_month_avg",
    "perf_12_month_avg",
    "negative_inventory_flag",
    "missing_lead_time_flag",
    "zero_lead_time_flag",
    "perf_6m_sentinel_flag",
    "perf_12m_sentinel_flag"
]

# Check whether every predictor exists
missing_features = [
    col for col in feature_cols
    if col not in df.columns
]

assert not missing_features, (
    f"Missing columns: {missing_features}"
)

print("Number of predictors:", len(feature_cols))
print("Predictor columns verified: PASS")


In [ ]:

# Predictor matrix for labelled SKUs
X = labelled_df[feature_cols].copy()

# Binary target: False = 0, True = 1
y = (
    labelled_df["went_on_backorder"]
    .astype(bool)
    .astype(int)
)

# Predictor matrix for unlabelled SKUs
X_unlabelled = unlabelled_df[feature_cols].copy()

# Preserve SKU identifiers separately
labelled_sku_ids = labelled_df["sku"].copy()
unlabelled_sku_ids = unlabelled_df["sku"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_unlabelled shape:", X_unlabelled.shape)

print("\nTarget distribution:")
print(y.value_counts().sort_index())

assert X.shape == (15243, 18)
assert y.shape == (15243,)
assert X_unlabelled.shape == (3810, 18)
assert y.value_counts().to_dict() == {0: 13430, 1: 1813}

print("\nPredictor and target preparation: PASS")


In [ ]:

# Phase 5 - Action 5.2.1C
# Stratified 80/20 development and holdout split

from sklearn.model_selection import train_test_split

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Development X shape:", X_dev.shape)
print("Holdout X shape:", X_holdout.shape)

print("\nDevelopment target distribution:")
print(y_dev.value_counts().sort_index())

print("\nHoldout target distribution:")
print(y_holdout.value_counts().sort_index())

print("\nDevelopment backorder rate:",
      f"{y_dev.mean():.2%}")

print("Holdout backorder rate:",
      f"{y_holdout.mean():.2%}")


In [ ]:

# Validate development / holdout separation

assert X_dev.shape == (12194, 18)
assert X_holdout.shape == (3049, 18)

assert len(y_dev) == 12194
assert len(y_holdout) == 3049

# Every labelled record must belong to exactly one partition
assert set(X_dev.index).isdisjoint(set(X_holdout.index))
assert set(X_dev.index) | set(X_holdout.index) == set(X.index)

# Confirm target-class proportions were preserved
overall_rate = y.mean()
dev_rate = y_dev.mean()
holdout_rate = y_holdout.mean()

assert abs(dev_rate - overall_rate) < 0.001
assert abs(holdout_rate - overall_rate) < 0.001

# Confirm unlabelled records remain separate
assert X_unlabelled.shape == (3810, 18)

print("Development/Holdout Split: PASS")
print("No record overlap: PASS")
print("Stratification: PASS")
print("Unlabelled dataset preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.1D
# Leakage-safe numerical preprocessing

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Preprocessing pipeline created successfully!")

print("\nPipeline steps:")
for name, step in numeric_preprocessor.steps:
    print(f"{name}: {step}")


In [ ]:

from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

# Confirm the preprocessor has not learned from any data yet
try:
    check_is_fitted(numeric_preprocessor)
    raise AssertionError("Preprocessor was unexpectedly fitted.")
except NotFittedError:
    print("Preprocessor is unfitted: PASS")

assert len(numeric_preprocessor.steps) == 2
assert numeric_preprocessor.steps[0][0] == "imputer"
assert numeric_preprocessor.steps[1][0] == "scaler"

print("Pipeline structure: PASS")
print("Holdout data has not been used: PASS")


In [ ]:

# Phase 5 - Action 5.2.2A
# Baseline Logistic Regression pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone

baseline_model = Pipeline(steps=[
    ("preprocessor", clone(numeric_preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("Baseline Logistic Regression pipeline created!")

print("\nPipeline steps:")
for name, step in baseline_model.steps:
    print(f"{name}: {step}")

print("\nClassifier settings:")
print("Algorithm:", baseline_model.named_steps["classifier"].__class__.__name__)
print("Maximum iterations:", baseline_model.named_steps["classifier"].max_iter)
print("Random state:", baseline_model.named_steps["classifier"].random_state)


In [ ]:

# Verify the baseline model structure

from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

assert len(baseline_model.steps) == 2

assert baseline_model.steps[0][0] == "preprocessor"
assert baseline_model.steps[1][0] == "classifier"

assert isinstance(
    baseline_model.named_steps["classifier"],
    LogisticRegression
)

assert baseline_model.named_steps["classifier"].max_iter == 2000
assert baseline_model.named_steps["classifier"].random_state == 42
assert baseline_model.named_steps["classifier"].class_weight is None

try:
    check_is_fitted(baseline_model)
    raise AssertionError("Model was unexpectedly fitted.")
except NotFittedError:
    print("Baseline model is unfitted: PASS")

print("Baseline model structure: PASS")
print("Classifier configuration: PASS")


In [ ]:

# Phase 5 - Action 5.2.2B
# Baseline Logistic Regression: 5-fold cross-validation

import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

baseline_cv = cross_validate(
    baseline_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Baseline cross-validation completed!")


In [ ]:

# Summarize the five validation folds

metric_names = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

results = []

for metric in metric_names:
    scores = baseline_cv[f"test_{metric}"]

    results.append({
        "Metric": metric,
        "Mean": scores.mean(),
        "Std Dev": scores.std(),
        "Minimum": scores.min(),
        "Maximum": scores.max()
    })

baseline_summary = pd.DataFrame(results)

display(baseline_summary.round(4))

print("\nPrimary metric: Average Precision")
print(
    "Mean Average Precision:",
    round(baseline_cv["test_average_precision"].mean(), 4)
)


In [ ]:

# Cross-validation verification

assert len(baseline_cv["test_average_precision"]) == 5
assert len(baseline_cv["test_roc_auc"]) == 5
assert len(baseline_cv["test_precision"]) == 5
assert len(baseline_cv["test_recall"]) == 5
assert len(baseline_cv["test_f1"]) == 5

for metric in metric_names:
    scores = baseline_cv[f"test_{metric}"]
    assert ((scores >= 0) & (scores <= 1)).all(), (
        f"Invalid scores for {metric}"
    )

assert X_holdout.shape == (3049, 18)

print("Five-fold evaluation: PASS")
print("Metric ranges: PASS")
print("Holdout dataset preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.2C
# Baseline performance analysis

import pandas as pd

# Historical backorder prevalence
backorder_prevalence = y_dev.mean()

# Cross-validation results
mean_ap = baseline_cv["test_average_precision"].mean()
mean_roc_auc = baseline_cv["test_roc_auc"].mean()
mean_precision = baseline_cv["test_precision"].mean()
mean_recall = baseline_cv["test_recall"].mean()
mean_f1 = baseline_cv["test_f1"].mean()

print("BASELINE MODEL PERFORMANCE")
print("-" * 45)

print(f"Historical Backorder Prevalence: {backorder_prevalence:.2%}")
print(f"Average Precision (PR-AUC):    {mean_ap:.4f}")
print(f"ROC-AUC:                       {mean_roc_auc:.4f}")
print(f"Precision:                     {mean_precision:.4f}")
print(f"Recall:                        {mean_recall:.4f}")
print(f"F1-Score:                      {mean_f1:.4f}")

print("\nAVERAGE PRECISION COMPARISON")
print("-" * 45)

print(f"Reference prevalence: {backorder_prevalence:.4f}")
print(f"Model Average Precision: {mean_ap:.4f}")

print(f"AP / Prevalence: {mean_ap / backorder_prevalence:.2f}x")


In [ ]:

# Save cross-validation summary

from pathlib import Path

# Locate project root
project_root = Path.cwd()

if project_root.name.lower() == "python":
    project_root = project_root.parent

output_dir = project_root / "python" / "model_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_logistic_regression_cv.csv"

baseline_summary.to_csv(output_file, index=False)

print("Saved baseline results to:")
print(output_file)

assert output_file.exists()

print("\nBaseline evaluation export: PASS")


In [ ]:

# Phase 5 - Action 5.2.2D
# Class-weighted Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

weighted_model = Pipeline(steps=[
    ("preprocessor", clone(numeric_preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Class-weighted model created!")

print("Algorithm:",
      weighted_model.named_steps["classifier"].__class__.__name__)

print("Class weighting:",
      weighted_model.named_steps["classifier"].class_weight)

print("Maximum iterations:",
      weighted_model.named_steps["classifier"].max_iter)


In [ ]:

# Evaluate weighted model using the same CV setup

from sklearn.model_selection import cross_validate

weighted_cv = cross_validate(
    weighted_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Weighted Logistic Regression CV: PASS")


In [ ]:

# Compare baseline and class-weighted models

import pandas as pd

metrics = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

comparison = pd.DataFrame({
    "Metric": metrics,
    "Baseline": [
        baseline_cv[f"test_{m}"].mean()
        for m in metrics
    ],
    "Class Weighted": [
        weighted_cv[f"test_{m}"].mean()
        for m in metrics
    ]
})

comparison["Difference"] = (
    comparison["Class Weighted"] - comparison["Baseline"]
)

display(comparison.round(4))


In [ ]:

# Verify weighted model evaluation

import numpy as np

assert len(weighted_cv["test_average_precision"]) == 5

for metric in metrics:
    scores = weighted_cv[f"test_{metric}"]
    assert len(scores) == 5
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

assert weighted_model.named_steps[
    "classifier"
].class_weight == "balanced"

assert comparison.shape == (5, 4)

print("Weighted model evaluation: PASS")
print("Baseline comparison: PASS")
print("Holdout not used in this action: PASS")


In [ ]:

# Phase 5 - Action 5.2.2E
# Compare Logistic Regression models

import pandas as pd

comparison_display = comparison.copy()

comparison_display["Difference"] = (
    comparison_display["Class Weighted"]
    - comparison_display["Baseline"]
)

print("LOGISTIC REGRESSION MODEL COMPARISON")
print("-" * 55)

display(comparison_display.round(4))

# Primary model selection metric
baseline_ap = baseline_cv["test_average_precision"].mean()
weighted_ap = weighted_cv["test_average_precision"].mean()

print("\nPRIMARY METRIC: AVERAGE PRECISION")
print(f"Baseline AP:       {baseline_ap:.4f}")
print(f"Class-Weighted AP: {weighted_ap:.4f}")

if weighted_ap > baseline_ap:
    preferred_lr = "Class-Weighted Logistic Regression"
elif baseline_ap > weighted_ap:
    preferred_lr = "Baseline Logistic Regression"
else:
    preferred_lr = "Tie"

print("\nBetter Logistic Regression candidate by AP:")
print(preferred_lr)


In [ ]:

# Save the Logistic Regression comparison

from pathlib import Path

project_root = Path.cwd()

if project_root.name.lower() == "python":
    project_root = project_root.parent

output_dir = project_root / "python" / "model_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = (
    output_dir
    / "logistic_regression_model_comparison.csv"
)

comparison_display.to_csv(output_file, index=False)

print("Saved comparison to:")
print(output_file)

assert output_file.exists()
assert comparison_display.shape == (5, 4)

print("\nLogistic Regression comparison export: PASS")


In [ ]:

# Phase 5 - Action 5.2.3A
# Random Forest Classification Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

random_forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created successfully!")

print("\nPipeline steps:")
for name, step in random_forest_model.steps:
    print(f"{name}: {step}")

print("\nRandom Forest configuration:")
rf = random_forest_model.named_steps["classifier"]

print("Number of trees:", rf.n_estimators)
print("Maximum depth:", rf.max_depth)
print("Minimum samples per leaf:", rf.min_samples_leaf)
print("Class weighting:", rf.class_weight)
print("Random state:", rf.random_state)
print("Parallel jobs:", rf.n_jobs)


In [ ]:

python -c "import sklearn; print('scikit-learn:', sklearn.__version__); from sklearn.ensemble import RandomForestClassifier; print('Random Forest import: PASS')"


In [ ]:

# Phase 5 - Action 5.2.3A
# Random Forest Classification Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("Random Forest import: PASS")

# Create Random Forest pipeline
random_forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("\nRandom Forest pipeline created successfully!")

# Display pipeline steps
print("\nPipeline steps:")
for name, step in random_forest_model.steps:
    print(f"{name}: {step}")

# Display classifier configuration
rf = random_forest_model.named_steps["classifier"]

print("\nRandom Forest configuration:")
print("Number of trees:", rf.n_estimators)
print("Maximum depth:", rf.max_depth)
print("Minimum samples per leaf:", rf.min_samples_leaf)
print("Class weighting:", rf.class_weight)
print("Random state:", rf.random_state)
print("Parallel jobs:", rf.n_jobs)

print("\nRandom Forest pipeline creation: PASS")


In [ ]:

# Phase 5 - Action 5.2.3B
# Random Forest five-fold cross-validation

from sklearn.model_selection import cross_validate

rf_cv = cross_validate(
    random_forest_model,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Random Forest five-fold CV: PASS")



In [ ]:

# Summarize Random Forest performance

import pandas as pd

rf_metrics = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

rf_summary = pd.DataFrame({
    "Metric": rf_metrics,
    "Mean": [
        rf_cv[f"test_{m}"].mean()
        for m in rf_metrics
    ],
    "Std Dev": [
        rf_cv[f"test_{m}"].std()
        for m in rf_metrics
    ]
})

print("RANDOM FOREST CROSS-VALIDATION RESULTS")
display(rf_summary.round(4))


In [ ]:

# Compare three classification models

model_comparison = pd.DataFrame({
    "Metric": rf_metrics,
    "Baseline LR": [
        baseline_cv[f"test_{m}"].mean()
        for m in rf_metrics
    ],
    "Weighted LR": [
        weighted_cv[f"test_{m}"].mean()
        for m in rf_metrics
    ],
    "Random Forest": [
        rf_cv[f"test_{m}"].mean()
        for m in rf_metrics
    ]
})

print("THREE-MODEL PERFORMANCE COMPARISON")
display(model_comparison.round(4))

# Identify the current AP leader
ap_row = model_comparison.loc[
    model_comparison["Metric"] == "average_precision"
].iloc[0]

ap_scores = ap_row.drop("Metric")

print("\nHighest mean Average Precision:")
print(ap_scores.idxmax(), round(ap_scores.max(), 4))



In [ ]:

# Validate Random Forest CV results

import numpy as np

assert len(rf_cv["test_average_precision"]) == 5

for metric in rf_metrics:
    scores = rf_cv[f"test_{metric}"]
    assert len(scores) == 5
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

assert rf_summary.shape == (5, 3)
assert model_comparison.shape == (5, 4)

assert X_dev.shape == (12194, 18)
assert X_holdout.shape == (3049, 18)

print("Random Forest evaluation: PASS")
print("Three-model comparison: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.3C
# Fit Random Forest for feature importance inspection

from sklearn.base import clone

rf_inspection_model = clone(random_forest_model)

rf_inspection_model.fit(X_dev, y_dev)

print("Random Forest inspection model fitted: PASS")
print("Development records used:", len(X_dev))
print("Holdout records used: 0")
print("Unlabelled records used: 0")


In [ ]:

# Extract Random Forest feature importance

import pandas as pd

rf_classifier = rf_inspection_model.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_classifier.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance["Rank"] = (
    range(1, len(feature_importance) + 1)
)

feature_importance = feature_importance[
    ["Rank", "Feature", "Importance"]
]

print("RANDOM FOREST FEATURE IMPORTANCE")
display(feature_importance.round(4))


In [ ]:

# Visualize Random Forest feature importance

import matplotlib.pyplot as plt

plot_data = feature_importance.sort_values(
    "Importance", ascending=True
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["Feature"],
    plot_data["Importance"]
)

plt.xlabel("Random Forest Feature Importance")
plt.ylabel("Inventory Predictor")
plt.title("Backorder Model - Feature Importance")

plt.tight_layout()
plt.show()


In [ ]:

# Verify feature importance and screen for known exclusions

import numpy as np

excluded_columns = {
    "sku",
    "dataset_source",
    "went_on_backorder",
    "pieces_past_due",
    "local_bo_qty",
    "potential_issue",
    "deck_risk",
    "oe_constraint",
    "ppap_risk",
    "stop_auto_buy",
    "rev_stop"
}

assert len(feature_importance) == 18
assert feature_importance["Feature"].is_unique

assert np.isfinite(
    feature_importance["Importance"]
).all()

assert np.isclose(
    feature_importance["Importance"].sum(), 1.0
)

assert not excluded_columns.intersection(feature_cols)

print("Feature importance integrity: PASS")
print("Known excluded columns absent: PASS")

print("\nTOP 5 FEATURES")
display(feature_importance.head(5).round(4))

print(
    "\nNOTE: Prediction-time availability "
    "still requires investigation."
)


In [ ]:

# Phase 5 - Action 5.2.3D
# Create a feature set excluding demand forecasts

forecast_features = [
    "forecast_3_month",
    "forecast_6_month",
    "forecast_9_month"
]

reduced_features = [
    col for col in feature_cols
    if col not in forecast_features
]

X_dev_reduced = X_dev[reduced_features].copy()

print("Original feature count:", len(feature_cols))
print("Reduced feature count:", len(reduced_features))

print("\nExcluded forecast variables:")
for col in forecast_features:
    print("-", col)

print("\nReduced development matrix:", X_dev_reduced.shape)

assert len(feature_cols) == 18
assert len(reduced_features) == 15
assert X_dev_reduced.shape == (12194, 15)

print("\nReduced feature set: PASS")


In [ ]:

# Evaluate Random Forest without forecast variables

from sklearn.base import clone
from sklearn.model_selection import cross_validate

rf_reduced_model = clone(random_forest_model)

rf_reduced_cv = cross_validate(
    rf_reduced_model,
    X_dev_reduced,
    y_dev,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Reduced Random Forest cross-validation: PASS")


In [ ]:

# Compare full and reduced Random Forest models

import pandas as pd

sensitivity_metrics = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

forecast_sensitivity = pd.DataFrame({
    "Metric": sensitivity_metrics,
    "RF - 18 Features": [
        rf_cv[f"test_{m}"].mean()
        for m in sensitivity_metrics
    ],
    "RF - 15 Features": [
        rf_reduced_cv[f"test_{m}"].mean()
        for m in sensitivity_metrics
    ]
})

forecast_sensitivity["Difference"] = (
    forecast_sensitivity["RF - 15 Features"]
    - forecast_sensitivity["RF - 18 Features"]
)

print("FORECAST FEATURE SENSITIVITY ANALYSIS")
display(forecast_sensitivity.round(4))


In [ ]:

# Validate reduced-model evaluation

import numpy as np

assert forecast_sensitivity.shape == (5, 4)

assert len(rf_reduced_cv["test_average_precision"]) == 5

for metric in sensitivity_metrics:
    scores = rf_reduced_cv[f"test_{metric}"]
    assert len(scores) == 5
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

assert not set(forecast_features).intersection(
    X_dev_reduced.columns
)

assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Reduced feature exclusion: PASS")
print("Sensitivity evaluation: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.3E
# Save Random Forest forecast sensitivity results

from pathlib import Path

project_root = Path.cwd()

if project_root.name.lower() == "python":
    project_root = project_root.parent

output_dir = project_root / "python" / "model_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

sensitivity_file = (
    output_dir / "random_forest_forecast_sensitivity.csv"
)

forecast_sensitivity.to_csv(
    sensitivity_file,
    index=False
)

assert sensitivity_file.exists()
assert forecast_sensitivity.shape == (5, 4)

print("Forecast sensitivity export: PASS")
print("Saved to:", sensitivity_file)


In [ ]:

# Audit the 15 remaining predictors

import pandas as pd

feature_notes = {
    "national_inv": (
        "Inventory snapshot timing must be verified"
    ),
    "in_transit_qty": (
        "Confirm quantities were known at prediction time"
    ),
    "min_bank": (
        "Confirm minimum stock threshold was predefined"
    ),
    "sales_1_month": (
        "Confirm sales history ended before outcome"
    ),
    "sales_3_month": (
        "Confirm sales history ended before outcome"
    ),
    "sales_6_month": (
        "Confirm sales history ended before outcome"
    ),
    "sales_9_month": (
        "Confirm sales history ended before outcome"
    ),
    "lead_time": (
        "Confirm lead time was known before outcome"
    ),
    "perf_6_month_avg": (
        "Verify supplier performance observation window"
    ),
    "perf_12_month_avg": (
        "Verify supplier performance observation window"
    ),
    "negative_inventory_flag": (
        "Derived from inventory; inherits snapshot timing"
    ),
    "missing_lead_time_flag": (
        "Derived from lead-time availability"
    ),
    "zero_lead_time_flag": (
        "Derived from lead-time value"
    ),
    "perf_6m_sentinel_flag": (
        "Derived from supplier-performance data quality"
    ),
    "perf_12m_sentinel_flag": (
        "Derived from supplier-performance data quality"
    )
}

feature_audit = pd.DataFrame({
    "Feature": reduced_features
})

feature_audit["Availability Status"] = (
    "Not independently verified"
)

feature_audit["Verification Required"] = (
    feature_audit["Feature"].map(feature_notes)
)

feature_audit["Include in Sensitivity Model"] = True

print("REMAINING FEATURE AVAILABILITY AUDIT")
display(feature_audit)


In [ ]:

# Export remaining-feature availability audit

audit_file = (
    output_dir / "random_forest_feature_availability_audit.csv"
)

feature_audit.to_csv(
    audit_file,
    index=False
)

assert audit_file.exists()

assert len(feature_audit) == 15
assert feature_audit["Feature"].is_unique
assert feature_audit["Verification Required"].notna().all()

assert set(feature_audit["Feature"]) == set(reduced_features)

print("Feature availability audit: PASS")
print("Feature audit export: PASS")
print("Saved to:", audit_file)


In [ ]:

# Phase 5 - Action 5.2.4A
# Random Forest robustness testing
# 3 repetitions of stratified 5-fold cross-validation

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate
)
from sklearn.base import clone

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

rf_robustness_results = cross_validate(
    clone(rf_reduced_model),
    X_dev_reduced,
    y_dev,
    cv=repeated_cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Repeated Random Forest CV: PASS")
print(
    "Validation runs:",
    len(rf_robustness_results["test_average_precision"])
)


In [ ]:

# Summarize robustness results across 15 folds

import pandas as pd

robustness_metrics = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

robustness_summary = pd.DataFrame([
    {
        "Metric": metric,
        "Mean": rf_robustness_results[
            f"test_{metric}"
        ].mean(),
        "Std Dev": rf_robustness_results[
            f"test_{metric}"
        ].std(),
        "Minimum": rf_robustness_results[
            f"test_{metric}"
        ].min(),
        "Maximum": rf_robustness_results[
            f"test_{metric}"
        ].max()
    }
    for metric in robustness_metrics
])

print("RANDOM FOREST ROBUSTNESS SUMMARY")
display(robustness_summary.round(4))


In [ ]:

# Compare original 5-fold CV with repeated 5-fold CV

robustness_comparison = pd.DataFrame({
    "Metric": robustness_metrics,
    "Original 5-Fold": [
        rf_reduced_cv[f"test_{m}"].mean()
        for m in robustness_metrics
    ],
    "Repeated 5-Fold": [
        rf_robustness_results[f"test_{m}"].mean()
        for m in robustness_metrics
    ]
})

robustness_comparison["Difference"] = (
    robustness_comparison["Repeated 5-Fold"]
    - robustness_comparison["Original 5-Fold"]
)

print("ORIGINAL VS REPEATED CV")
display(robustness_comparison.round(4))


In [ ]:

# Verify repeated cross-validation results

import numpy as np

for metric in robustness_metrics:
    scores = rf_robustness_results[f"test_{metric}"]

    assert len(scores) == 15
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

assert robustness_summary.shape == (5, 5)
assert robustness_comparison.shape == (5, 4)

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Fifteen validation runs: PASS")
print("Metric integrity: PASS")
print("Robustness comparison: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.4B
# Random Forest hyperparameter optimization

from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

# Candidate hyperparameter values
param_distributions = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [8, 12, 16, None],
    "classifier__min_samples_leaf": [2, 5, 10],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__max_features": ["sqrt", 0.7],
    "classifier__class_weight": [
        "balanced",
        "balanced_subsample",
        None
    ]
}

rf_search = RandomizedSearchCV(
    estimator=clone(rf_reduced_model),
    param_distributions=param_distributions,
    n_iter=12,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

print("Random Forest search configured: PASS")
print("Parameter combinations to test: 12")
print("CV folds per combination: 5")
print("Primary metric: Average Precision")


In [ ]:

# Run hyperparameter optimization
# Development data only

rf_search.fit(X_dev_reduced, y_dev)

print("\nRandom Forest optimization: PASS")

print("\nBest cross-validation Average Precision:")
print(round(rf_search.best_score_, 4))

print("\nBest hyperparameters:")
for parameter, value in rf_search.best_params_.items():
    print(f"{parameter}: {value}")

print("\nBest estimator refitted:", rf_search.refit)


In [ ]:

# Review top five Random Forest configurations

import pandas as pd

tuning_results = pd.DataFrame(
    rf_search.cv_results_
)

top_configs = (
    tuning_results
    .sort_values("rank_test_score")
    .head(5)
    [[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "params"
    ]]
)

print("TOP FIVE RANDOM FOREST CONFIGURATIONS")
display(top_configs)

print("\nOriginal RF Average Precision:",
      round(
          rf_reduced_cv["test_average_precision"].mean(),
          4
      ))

print("Best tuning CV Average Precision:",
      round(rf_search.best_score_, 4))


In [ ]:

# Validate Random Forest hyperparameter search

import numpy as np
from sklearn.utils.validation import check_is_fitted

assert len(rf_search.cv_results_["params"]) == 12

assert np.isfinite(rf_search.best_score_)

assert 0 <= rf_search.best_score_ <= 1

assert rf_search.best_estimator_.named_steps[
    "classifier"
].__class__.__name__ == "RandomForestClassifier"

check_is_fitted(rf_search.best_estimator_)

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)

print("Hyperparameter search integrity: PASS")
print("Best estimator fitted on development data: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.4C
# Evaluate tuned Random Forest with repeated CV

from sklearn.base import clone
from sklearn.model_selection import cross_validate

# Clone the selected configuration without fitted parameters
tuned_rf_model = clone(rf_search.best_estimator_)

# Use the same repeated CV splits as the original model
tuned_rf_robustness = cross_validate(
    tuned_rf_model,
    X_dev_reduced,
    y_dev,
    cv=repeated_cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

print("Tuned Random Forest repeated CV: PASS")
print(
    "Validation runs:",
    len(tuned_rf_robustness["test_average_precision"])
)


In [ ]:

# Compare original and tuned Random Forest models

import pandas as pd

comparison_metrics = [
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1"
]

rf_tuning_comparison = pd.DataFrame({
    "Metric": comparison_metrics,
    "Original RF": [
        rf_robustness_results[f"test_{m}"].mean()
        for m in comparison_metrics
    ],
    "Tuned RF": [
        tuned_rf_robustness[f"test_{m}"].mean()
        for m in comparison_metrics
    ]
})

rf_tuning_comparison["Difference"] = (
    rf_tuning_comparison["Tuned RF"]
    - rf_tuning_comparison["Original RF"]
)

print("ORIGINAL VS TUNED RANDOM FOREST")
display(rf_tuning_comparison.round(4))


In [ ]:

# Compare fold-by-fold Average Precision

import numpy as np

original_ap = rf_robustness_results[
    "test_average_precision"
]

tuned_ap = tuned_rf_robustness[
    "test_average_precision"
]

ap_differences = tuned_ap - original_ap

fold_comparison = pd.DataFrame({
    "Validation Run": range(1, 16),
    "Original AP": original_ap,
    "Tuned AP": tuned_ap,
    "Difference": ap_differences
})

print("FOLD-BY-FOLD AVERAGE PRECISION")
display(fold_comparison.round(4))

print("\nAverage AP difference:",
      round(ap_differences.mean(), 4))

print("Runs where tuned RF performed better:",
      int((ap_differences > 0).sum()))

print("Runs where original RF performed better:",
      int((ap_differences < 0).sum()))


In [ ]:

# Validate original vs tuned model comparison

assert len(original_ap) == 15
assert len(tuned_ap) == 15

assert np.isfinite(original_ap).all()
assert np.isfinite(tuned_ap).all()

for metric in comparison_metrics:
    original_scores = rf_robustness_results[f"test_{metric}"]
    tuned_scores = tuned_rf_robustness[f"test_{metric}"]

    assert len(original_scores) == 15
    assert len(tuned_scores) == 15

    assert np.isfinite(original_scores).all()
    assert np.isfinite(tuned_scores).all()

assert rf_tuning_comparison.shape == (5, 4)
assert fold_comparison.shape == (15, 4)

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Paired validation runs: PASS")
print("Model comparison integrity: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.4D
# Export original vs tuned Random Forest comparison

from pathlib import Path

project_root = Path.cwd()

if project_root.name.lower() == "python":
    project_root = project_root.parent

output_dir = project_root / "python" / "model_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_file = (
    output_dir / "random_forest_original_vs_tuned.csv"
)

fold_file = (
    output_dir / "random_forest_fold_comparison.csv"
)

rf_tuning_comparison.to_csv(
    comparison_file, index=False
)

fold_comparison.to_csv(
    fold_file, index=False
)

assert comparison_file.exists()
assert fold_file.exists()

print("Model comparison export: PASS")
print("Fold comparison export: PASS")

print("\nSaved files:")
print(comparison_file.name)
print(fold_file.name)


In [ ]:

# Record preferred Random Forest configuration

import json
from datetime import datetime, timezone

selected_model = {
    "model_name": "Random Forest",
    "model_stage": "development_candidate",
    "feature_set": "15 features without forecasts",
    "feature_columns": reduced_features,
    "selection_metric": "average_precision",
    "validation_method": "RepeatedStratifiedKFold",
    "n_splits": 5,
    "n_repeats": 3,
    "random_state": 42,
    "model_parameters": {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "min_samples_split": 2,
        "max_features": "sqrt",
        "class_weight": "balanced"
    },
    "cv_performance": {
        "average_precision": float(
            rf_robustness_results[
                "test_average_precision"
            ].mean()
        ),
        "roc_auc": float(
            rf_robustness_results["test_roc_auc"].mean()
        ),
        "precision": float(
            rf_robustness_results["test_precision"].mean()
        ),
        "recall": float(
            rf_robustness_results["test_recall"].mean()
        ),
        "f1": float(
            rf_robustness_results["test_f1"].mean()
        )
    },
    "holdout_evaluated": False,
    "prediction_time_availability_verified": False,
    "limitation": (
        "Historical classification only; "
        "prediction timestamps are unavailable."
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

selection_file = (
    output_dir / "selected_model_configuration.json"
)

with open(selection_file, "w", encoding="utf-8") as f:
    json.dump(selected_model, f, indent=4)

print("Model selection documentation: PASS")
print("Saved:", selection_file.name)


In [ ]:

# Verify development-stage model selection

with open(selection_file, "r", encoding="utf-8") as f:
    saved_selection = json.load(f)

assert saved_selection["model_name"] == "Random Forest"
assert saved_selection["model_stage"] == "development_candidate"

assert len(saved_selection["feature_columns"]) == 15

assert not any(
    f.startswith("forecast_")
    for f in saved_selection["feature_columns"]
)

assert saved_selection["model_parameters"][
    "n_estimators"
] == 200

assert saved_selection["model_parameters"][
    "max_depth"
] == 12

assert saved_selection["holdout_evaluated"] is False

assert saved_selection[
    "prediction_time_availability_verified"
] is False

assert comparison_file.exists()
assert fold_file.exists()
assert selection_file.exists()

print("Selected model configuration: PASS")
print("Model selection documentation: PASS")
print("Exported files verification: PASS")
print("Holdout status correctly documented: PASS")


In [ ]:

# Phase 5 - Action 5.2.5A
# Generate out-of-fold backorder probabilities

import numpy as np
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    estimator=clone(rf_reduced_model),
    X=X_dev_reduced,
    y=y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Out-of-fold probabilities generated: PASS")
print("Development records:", len(oof_probabilities))

print("\nProbability summary:")
print("Minimum:", round(oof_probabilities.min(), 4))
print("Maximum:", round(oof_probabilities.max(), 4))
print("Mean:", round(oof_probabilities.mean(), 4))

assert len(oof_probabilities) == 12194
assert np.isfinite(oof_probabilities).all()
assert ((oof_probabilities >= 0) &
        (oof_probabilities <= 1)).all()

print("\nOut-of-fold probability integrity: PASS")


In [ ]:

# Analyze precision and recall at different thresholds

import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = np.round(
    np.arange(0.10, 0.91, 0.05), 2
)

threshold_results = []

for threshold in thresholds:

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_dev,
        predictions,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_dev, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_dev, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_dev, predictions, zero_division=0
        ),
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "True Negatives": tn,
        "Flagged SKUs": tp + fp
    })

threshold_analysis = pd.DataFrame(
    threshold_results
)

print("PRECISION-RECALL THRESHOLD ANALYSIS")
display(threshold_analysis.round(4))


In [ ]:

# Plot precision, recall and F1 against threshold

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    threshold_analysis["Threshold"],
    threshold_analysis["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_analysis["Threshold"],
    threshold_analysis["Recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_analysis["Threshold"],
    threshold_analysis["F1"],
    marker="o",
    label="F1 Score"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Random Forest - Threshold Analysis")

plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Compare default threshold with highest-F1 candidate

default_row = threshold_analysis.loc[
    np.isclose(
        threshold_analysis["Threshold"], 0.50
    )
].iloc[0]

best_f1_row = threshold_analysis.loc[
    threshold_analysis["F1"].idxmax()
]

print("DEFAULT THRESHOLD")
print(default_row.to_string())

print("\nHIGHEST-F1 THRESHOLD (EXPLORATORY)")
print(best_f1_row.to_string())

print("\nDefault threshold:",
      round(default_row["Threshold"], 2))

print("Highest-F1 threshold:",
      round(best_f1_row["Threshold"], 2))

print("\nThreshold exploration: PASS")


In [ ]:

# Verify threshold analysis integrity

assert len(threshold_analysis) == 17

assert np.isclose(
    threshold_analysis["Threshold"].iloc[0],
    0.10
)

assert np.isclose(
    threshold_analysis["Threshold"].iloc[-1],
    0.90
)

assert threshold_analysis[
    ["Precision", "Recall", "F1"]
].apply(
    lambda s: s.between(0, 1).all()
).all()

assert (
    threshold_analysis[
        [
            "True Positives",
            "False Positives",
            "False Negatives",
            "True Negatives"
        ]
    ].sum(axis=1) == 12194
).all()

assert (
    threshold_analysis["Flagged SKUs"]
    == threshold_analysis["True Positives"]
    + threshold_analysis["False Positives"]
).all()

assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Threshold count: PASS")
print("Classification metrics: PASS")
print("Confusion matrix totals: PASS")
print("Flagged SKU counts: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.5B
# Operational threshold comparison

import pandas as pd
import numpy as np

candidate_thresholds = [0.50, 0.60, 0.65, 0.70]

business_comparison = threshold_analysis[
    threshold_analysis["Threshold"].isin(
        candidate_thresholds
    )
].copy()

business_comparison["Alert Rate (%)"] = (
    business_comparison["Flagged SKUs"]
    / len(y_dev) * 100
)

business_comparison["Missed Backorder Rate (%)"] = (
    business_comparison["False Negatives"]
    / (business_comparison["True Positives"]
       + business_comparison["False Negatives"])
    * 100
)

print("BUSINESS THRESHOLD COMPARISON")

display(
    business_comparison[
        [
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "Flagged SKUs",
            "False Positives",
            "False Negatives",
            "Alert Rate (%)",
            "Missed Backorder Rate (%)"
        ]
    ].round(4)
)

print("\nOperational threshold comparison: PASS")


In [ ]:

# Compare relative business costs
# These are hypothetical cost weights, not actual costs.

cost_scenarios = {
    "Equal Cost (1:1)": 1,
    "Moderate Shortage Cost (3:1)": 3,
    "High Shortage Cost (5:1)": 5
}

cost_results = []

for scenario, missed_cost in cost_scenarios.items():

    for _, row in threshold_analysis.iterrows():

        relative_cost = (
            row["False Positives"] * 1
            + row["False Negatives"] * missed_cost
        )

        cost_results.append({
            "Scenario": scenario,
            "Threshold": row["Threshold"],
            "False Positives": row["False Positives"],
            "False Negatives": row["False Negatives"],
            "Relative Cost": relative_cost
        })

cost_analysis = pd.DataFrame(cost_results)

best_cost_thresholds = (
    cost_analysis
    .sort_values(["Scenario", "Relative Cost", "Threshold"])
    .groupby("Scenario", as_index=False)
    .first()
)

print("LOWEST-COST THRESHOLD BY SCENARIO")
display(best_cost_thresholds)

print("\nBusiness cost scenarios: PASS")


In [ ]:

# Plot relative costs across classification thresholds

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for scenario in cost_scenarios:
    scenario_data = cost_analysis[
        cost_analysis["Scenario"] == scenario
    ]

    plt.plot(
        scenario_data["Threshold"],
        scenario_data["Relative Cost"],
        marker="o",
        label=scenario
    )

plt.xlabel("Classification Threshold")
plt.ylabel("Relative Misclassification Cost")
plt.title("Random Forest - Business Cost Sensitivity")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Phase 5 - Action 5.2.5B
# Business threshold validation

assert business_comparison.shape[0] == 4
assert cost_analysis.shape[0] == 51
assert best_cost_thresholds.shape[0] == 3

assert (
    business_comparison["Alert Rate (%)"]
    .between(0, 100)
).all()

assert (
    business_comparison["Missed Backorder Rate (%)"]
    .between(0, 100)
).all()

assert (cost_analysis["Relative Cost"] >= 0).all()

for scenario in cost_scenarios:
    subset = cost_analysis[
        cost_analysis["Scenario"] == scenario
    ]
    recorded_min = best_cost_thresholds.loc[
        best_cost_thresholds["Scenario"] == scenario,
        "Relative Cost"
    ].iloc[0]

    assert np.isclose(
        recorded_min,
        subset["Relative Cost"].min()
    )

assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Operational comparison: PASS")
print("Cost scenario calculations: PASS")
print("Minimum-cost thresholds: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.5C
# Provisional operating threshold

import numpy as np

provisional_threshold = 0.60

# Retrieve historical out-of-fold performance
selected_row = threshold_analysis.loc[
    np.isclose(
        threshold_analysis["Threshold"],
        provisional_threshold
    )
].iloc[0]

print("PROVISIONAL THRESHOLD")
print("-" * 45)

print("Threshold:", provisional_threshold)
print(f"Precision: {selected_row['Precision']:.2%}")
print(f"Recall: {selected_row['Recall']:.2%}")
print(f"F1-score: {selected_row['F1']:.4f}")

print("True positives:",
      int(selected_row["True Positives"]))

print("False positives:",
      int(selected_row["False Positives"]))

print("False negatives:",
      int(selected_row["False Negatives"]))

print("True negatives:",
      int(selected_row["True Negatives"]))

print("Flagged SKUs:",
      int(selected_row["Flagged SKUs"]))

print("\nProvisional threshold selected: PASS")


In [ ]:

# Document business reasoning and limitations

threshold_documentation = {
    "model": "Original 15-feature Random Forest",
    "threshold_status": "provisional",
    "selected_threshold": float(provisional_threshold),
    "selection_scenario": "Moderate shortage cost (3:1)",
    "false_positive_cost_weight": 1,
    "false_negative_cost_weight": 3,
    "business_justification": (
        "Selected as the lowest-cost tested threshold "
        "under a hypothetical 3:1 relative penalty "
        "for missed backorders versus false alarms."
    ),
    "precision": float(selected_row["Precision"]),
    "recall": float(selected_row["Recall"]),
    "f1_score": float(selected_row["F1"]),
    "true_positives": int(selected_row["True Positives"]),
    "false_positives": int(selected_row["False Positives"]),
    "false_negatives": int(selected_row["False Negatives"]),
    "true_negatives": int(selected_row["True Negatives"]),
    "flagged_skus": int(selected_row["Flagged SKUs"]),
    "limitations": [
        "Business-cost weights are hypothetical.",
        "Threshold was selected using development data.",
        "Reported threshold performance is exploratory.",
        "Probability calibration has not been validated.",
        "Prediction-time feature availability is unverified.",
        "Final holdout dataset remains unevaluated."
    ]
}

print("THRESHOLD DOCUMENTATION")
for key, value in threshold_documentation.items():
    print(f"{key}: {value}")

print("\nThreshold documentation prepared: PASS")


In [ ]:

# Export provisional threshold documentation

import json
from pathlib import Path

threshold_file = (
    output_dir / "provisional_threshold_selection.json"
)

with open(threshold_file, "w", encoding="utf-8") as f:
    json.dump(
        threshold_documentation,
        f,
        indent=4
    )

assert threshold_file.exists()

print("Threshold documentation export: PASS")
print("Saved to:", threshold_file)


In [ ]:

# Verify provisional threshold selection

# Confirm 0.60 was preferred under the 3:1 scenario
moderate_scenario = best_cost_thresholds.loc[
    best_cost_thresholds["Scenario"]
    == "Moderate Shortage Cost (3:1)"
].iloc[0]

assert np.isclose(
    moderate_scenario["Threshold"],
    provisional_threshold
)

assert selected_row["True Positives"] == 1152
assert selected_row["False Positives"] == 775
assert selected_row["False Negatives"] == 298
assert selected_row["True Negatives"] == 9969

assert selected_row["Flagged SKUs"] == 1927

assert threshold_documentation[
    "threshold_status"
] == "provisional"

assert threshold_documentation[
    "selected_threshold"
] == 0.60

assert threshold_file.exists()

# Verify existing dataset dimensions
assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Business scenario alignment: PASS")
print("Threshold metrics integrity: PASS")
print("Provisional documentation: PASS")
print("Threshold file export: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.6A
# Assess Random Forest probability calibration

import numpy as np

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    average_precision_score
)

actual_labels = y_dev.to_numpy()

brier_score = brier_score_loss(
    actual_labels,
    oof_probabilities
)

logloss_score = log_loss(
    actual_labels,
    oof_probabilities,
    labels=[0, 1]
)

oof_ap = average_precision_score(
    actual_labels,
    oof_probabilities
)

actual_backorder_rate = actual_labels.mean()
mean_predicted_probability = oof_probabilities.mean()

print("RANDOM FOREST CALIBRATION METRICS")
print("-" * 48)

print(f"Brier Score: {brier_score:.4f}")
print(f"Log Loss: {logloss_score:.4f}")
print(f"OOF Average Precision: {oof_ap:.4f}")

print(
    f"Actual backorder rate: "
    f"{actual_backorder_rate:.2%}"
)

print(
    f"Mean predicted probability: "
    f"{mean_predicted_probability:.2%}"
)

print(
    f"Mean probability gap: "
    f"{mean_predicted_probability - actual_backorder_rate:.2%}"
)

print("\nCalibration metrics calculated: PASS")


In [ ]:

# Compare Random Forest against prevalence-only reference

reference_probabilities = np.full(
    len(actual_labels),
    actual_backorder_rate
)

reference_brier = brier_score_loss(
    actual_labels,
    reference_probabilities
)

reference_logloss = log_loss(
    actual_labels,
    reference_probabilities,
    labels=[0, 1]
)

print("PROBABILITY QUALITY COMPARISON")
print("-" * 48)

print(f"Reference Brier Score: {reference_brier:.4f}")
print(f"Random Forest Brier Score: {brier_score:.4f}")

print(f"\nReference Log Loss: {reference_logloss:.4f}")
print(f"Random Forest Log Loss: {logloss_score:.4f}")

print("\nProbability reference comparison: PASS")


In [ ]:

# Create Random Forest calibration curve

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted_value = (
    calibration_curve(
        actual_labels,
        oof_probabilities,
        n_bins=10,
        strategy="uniform"
    )
)

plt.figure(figsize=(8, 7))

# Perfect calibration reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

# Actual Random Forest calibration
plt.plot(
    mean_predicted_value,
    fraction_of_positives,
    marker="o",
    label="Random Forest"
)

plt.xlabel("Mean Predicted Backorder Probability")
plt.ylabel("Observed Backorder Frequency")
plt.title("Random Forest - Probability Calibration")

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Calibration curve generated: PASS")


In [ ]:

# Probability-bin calibration audit

import pandas as pd

calibration_df = pd.DataFrame({
    "Actual": actual_labels,
    "Predicted Probability": oof_probabilities
})

bin_edges = np.linspace(0, 1, 11)

calibration_df["Probability Bin"] = pd.cut(
    calibration_df["Predicted Probability"],
    bins=bin_edges,
    include_lowest=True
)

calibration_summary = (
    calibration_df
    .groupby("Probability Bin", observed=False)
    .agg(
        SKU_Count=("Actual", "size"),
        Mean_Predicted=("Predicted Probability", "mean"),
        Actual_Backorder_Rate=("Actual", "mean")
    )
    .reset_index()
)

calibration_summary["Calibration_Gap"] = (
    calibration_summary["Mean_Predicted"]
    - calibration_summary["Actual_Backorder_Rate"]
)

print("PROBABILITY BIN CALIBRATION AUDIT")
display(calibration_summary.round(4))


In [ ]:

# Verify probability calibration assessment

assert len(actual_labels) == 12194
assert len(oof_probabilities) == 12194

assert np.isfinite([
    brier_score,
    logloss_score,
    oof_ap,
    reference_brier,
    reference_logloss
]).all()

assert 0 <= brier_score <= 1
assert 0 <= reference_brier <= 1

assert 0 <= oof_ap <= 1

assert calibration_summary["SKU_Count"].sum() == 12194

assert np.isclose(
    actual_backorder_rate,
    y_dev.mean()
)

assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Calibration metrics integrity: PASS")
print("Calibration bin totals: PASS")
print("Development label consistency: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.6B
# Create a sigmoid-calibrated Random Forest

from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

calibrated_rf_model = CalibratedClassifierCV(
    estimator=clone(rf_reduced_model),
    method="sigmoid",
    cv=inner_cv,
    ensemble=False
)

print("Calibrated Random Forest created: PASS")
print("Calibration method:", calibrated_rf_model.method)
print("Internal CV folds:", inner_cv.n_splits)
print("Ensemble setting:", calibrated_rf_model.ensemble)


In [ ]:

# Generate calibrated out-of-fold predictions
# Outer CV: 5 folds
# Inner calibration CV: 3 folds

import numpy as np
from sklearn.model_selection import cross_val_predict

calibrated_oof_probabilities = cross_val_predict(
    estimator=clone(calibrated_rf_model),
    X=X_dev_reduced,
    y=y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Calibrated OOF predictions generated: PASS")
print("Development records:", len(calibrated_oof_probabilities))

print(
    "Mean calibrated probability:",
    f"{calibrated_oof_probabilities.mean():.2%}"
)

assert len(calibrated_oof_probabilities) == 12194
assert np.isfinite(calibrated_oof_probabilities).all()
assert ((calibrated_oof_probabilities >= 0) &
        (calibrated_oof_probabilities <= 1)).all()

print("Calibrated probability integrity: PASS")


In [ ]:

# Compare probability quality before and after calibration

import pandas as pd
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    average_precision_score,
    roc_auc_score
)

calibration_comparison = pd.DataFrame({
    "Metric": [
        "Brier Score",
        "Log Loss",
        "Average Precision",
        "ROC-AUC",
        "Mean Predicted Probability"
    ],
    "Uncalibrated": [
        brier_score_loss(actual_labels, oof_probabilities),
        log_loss(actual_labels, oof_probabilities),
        average_precision_score(
            actual_labels, oof_probabilities
        ),
        roc_auc_score(
            actual_labels, oof_probabilities
        ),
        oof_probabilities.mean()
    ],
    "Sigmoid Calibrated": [
        brier_score_loss(
            actual_labels, calibrated_oof_probabilities
        ),
        log_loss(
            actual_labels, calibrated_oof_probabilities
        ),
        average_precision_score(
            actual_labels, calibrated_oof_probabilities
        ),
        roc_auc_score(
            actual_labels, calibrated_oof_probabilities
        ),
        calibrated_oof_probabilities.mean()
    ]
})

calibration_comparison["Difference"] = (
    calibration_comparison["Sigmoid Calibrated"]
    - calibration_comparison["Uncalibrated"]
)

print("CALIBRATED VS UNCALIBRATED RANDOM FOREST")
display(calibration_comparison.round(4))

print("\nProbability comparison: PASS")


In [ ]:

# Plot calibrated vs uncalibrated reliability curves

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

uncal_actual, uncal_pred = calibration_curve(
    actual_labels,
    oof_probabilities,
    n_bins=10,
    strategy="uniform"
)

cal_actual, cal_pred = calibration_curve(
    actual_labels,
    calibrated_oof_probabilities,
    n_bins=10,
    strategy="uniform"
)

plt.figure(figsize=(8, 7))

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.plot(
    uncal_pred,
    uncal_actual,
    marker="o",
    label="Uncalibrated RF"
)

plt.plot(
    cal_pred,
    cal_actual,
    marker="o",
    label="Sigmoid Calibrated RF"
)

plt.xlabel("Mean Predicted Backorder Probability")
plt.ylabel("Observed Backorder Frequency")
plt.title("Random Forest - Probability Calibration Comparison")

plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Calibration comparison chart: PASS")


In [ ]:

# Phase 5 - Action 5.2.6B
# Validate sigmoid calibration assessment

assert calibration_comparison.shape == (5, 4)

assert len(calibrated_oof_probabilities) == 12194

assert np.isfinite(
    calibration_comparison[
        ["Uncalibrated", "Sigmoid Calibrated", "Difference"]
    ].to_numpy()
).all()

assert calibration_comparison.loc[
    calibration_comparison["Metric"] == "Brier Score",
    "Sigmoid Calibrated"
].iloc[0] >= 0

assert calibration_comparison.loc[
    calibration_comparison["Metric"] == "Brier Score",
    "Sigmoid Calibrated"
].iloc[0] <= 1

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Calibrated probability count: PASS")
print("Metric comparison integrity: PASS")
print("Calibration configuration: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.6C
# Calibrated threshold analysis

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

calibrated_thresholds = np.round(
    np.arange(0.05, 0.91, 0.05), 2
)

calibrated_threshold_results = []

for threshold in calibrated_thresholds:

    predictions = (
        calibrated_oof_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    calibrated_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            actual_labels, predictions, zero_division=0
        ),
        "Recall": recall_score(
            actual_labels, predictions, zero_division=0
        ),
        "F1": f1_score(
            actual_labels, predictions, zero_division=0
        ),
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "True Negatives": tn,
        "Flagged SKUs": tp + fp,
        "Alert Rate (%)": (
            (tp + fp) / len(actual_labels) * 100
        )
    })

calibrated_threshold_analysis = pd.DataFrame(
    calibrated_threshold_results
)

print("CALIBRATED THRESHOLD ANALYSIS")
display(calibrated_threshold_analysis.round(4))

print("\nCalibrated threshold analysis: PASS")


In [ ]:

# Business-cost sensitivity using calibrated predictions

calibrated_cost_results = []

for scenario, missed_cost in cost_scenarios.items():

    for _, row in calibrated_threshold_analysis.iterrows():

        relative_cost = (
            row["False Positives"]
            + missed_cost * row["False Negatives"]
        )

        calibrated_cost_results.append({
            "Scenario": scenario,
            "Threshold": row["Threshold"],
            "False Positives": int(row["False Positives"]),
            "False Negatives": int(row["False Negatives"]),
            "Relative Cost": int(relative_cost)
        })

calibrated_cost_analysis = pd.DataFrame(
    calibrated_cost_results
)

best_calibrated_thresholds = (
    calibrated_cost_analysis
    .sort_values(
        ["Scenario", "Relative Cost", "Threshold"]
    )
    .groupby("Scenario", as_index=False)
    .first()
)

print("BEST CALIBRATED THRESHOLD BY COST SCENARIO")
display(best_calibrated_thresholds)

print("\nCalibrated business-cost analysis: PASS")


In [ ]:

# Select provisional calibrated threshold

moderate_calibrated = best_calibrated_thresholds.loc[
    best_calibrated_thresholds["Scenario"]
    == "Moderate Shortage Cost (3:1)"
].iloc[0]

provisional_calibrated_threshold = float(
    moderate_calibrated["Threshold"]
)

selected_calibrated_row = (
    calibrated_threshold_analysis.loc[
        np.isclose(
            calibrated_threshold_analysis["Threshold"],
            provisional_calibrated_threshold
        )
    ].iloc[0]
)

print("PROVISIONAL CALIBRATED THRESHOLD")
print("-" * 45)

print(
    "Threshold:",
    provisional_calibrated_threshold
)

print(
    "Precision:",
    f"{selected_calibrated_row['Precision']:.2%}"
)

print(
    "Recall:",
    f"{selected_calibrated_row['Recall']:.2%}"
)

print(
    "F1-score:",
    f"{selected_calibrated_row['F1']:.4f}"
)

print(
    "Flagged SKUs:",
    int(selected_calibrated_row["Flagged SKUs"])
)

print(
    "False positives:",
    int(selected_calibrated_row["False Positives"])
)

print(
    "False negatives:",
    int(selected_calibrated_row["False Negatives"])
)

print("\nCalibrated threshold candidate selected: PASS")


In [ ]:

# Compare uncalibrated and calibrated provisional rules

threshold_rule_comparison = pd.DataFrame([
    {
        "Model": "Uncalibrated RF",
        "Threshold": 0.60,
        "Precision": float(selected_row["Precision"]),
        "Recall": float(selected_row["Recall"]),
        "F1": float(selected_row["F1"]),
        "Flagged SKUs": int(selected_row["Flagged SKUs"]),
        "False Positives": int(selected_row["False Positives"]),
        "False Negatives": int(selected_row["False Negatives"])
    },
    {
        "Model": "Sigmoid Calibrated RF",
        "Threshold": provisional_calibrated_threshold,
        "Precision": float(selected_calibrated_row["Precision"]),
        "Recall": float(selected_calibrated_row["Recall"]),
        "F1": float(selected_calibrated_row["F1"]),
        "Flagged SKUs": int(selected_calibrated_row["Flagged SKUs"]),
        "False Positives": int(
            selected_calibrated_row["False Positives"]
        ),
        "False Negatives": int(
            selected_calibrated_row["False Negatives"]
        )
    }
])

print("OLD VS CALIBRATED THRESHOLD RULE")
display(threshold_rule_comparison.round(4))


In [ ]:

# Verify calibrated threshold reassessment

assert calibrated_threshold_analysis.shape[0] == 18
assert calibrated_cost_analysis.shape[0] == 54
assert best_calibrated_thresholds.shape[0] == 3

assert (
    calibrated_threshold_analysis[
        [
            "True Positives",
            "False Positives",
            "False Negatives",
            "True Negatives"
        ]
    ].sum(axis=1) == 12194
).all()

assert calibrated_threshold_analysis[
    ["Precision", "Recall", "F1"]
].apply(
    lambda s: s.between(0, 1).all()
).all()

assert threshold_rule_comparison.shape == (2, 8)

assert np.isclose(
    selected_calibrated_row["Threshold"],
    provisional_calibrated_threshold
)

assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Calibrated threshold count: PASS")
print("Cost scenario integrity: PASS")
print("Provisional calibrated threshold: PASS")
print("Decision-rule comparison: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.2.6D
# Export calibration and calibrated threshold results

from pathlib import Path

project_root = Path.cwd()

if project_root.name.lower() == "python":
    project_root = project_root.parent

output_dir = project_root / "python" / "model_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

export_tables = {
    "random_forest_calibration_comparison.csv":
        calibration_comparison,

    "calibrated_threshold_analysis.csv":
        calibrated_threshold_analysis,

    "calibrated_business_cost_scenarios.csv":
        best_calibrated_thresholds,

    "calibrated_vs_uncalibrated_threshold.csv":
        threshold_rule_comparison
}

for filename, dataframe in export_tables.items():

    file_path = output_dir / filename

    dataframe.to_csv(file_path, index=False)

    assert file_path.exists()

    print("Saved:", filename)

print("\nCalibration and threshold exports: PASS")


In [ ]:

# Document the selected calibration and threshold

import json
from datetime import datetime, timezone

calibration_decision = {
    "model": "Random Forest",
    "model_stage": "development_candidate",
    "feature_count": len(reduced_features),
    "feature_columns": list(reduced_features),

    "calibration_method": "sigmoid",
    "inner_calibration_folds": 3,
    "outer_evaluation_folds": 5,

    "calibration_metrics": {
        "brier_score": 0.0532,
        "log_loss": 0.1797,
        "average_precision": 0.7310
    },

    "threshold_status": "provisional",

    "selected_threshold": float(
        provisional_calibrated_threshold
    ),

    "threshold_selection_scenario": (
        "Hypothetical moderate shortage cost (3:1)"
    ),

    "threshold_metrics": {
        "precision": float(
            selected_calibrated_row["Precision"]
        ),
        "recall": float(
            selected_calibrated_row["Recall"]
        ),
        "f1": float(
            selected_calibrated_row["F1"]
        ),
        "true_positives": int(
            selected_calibrated_row["True Positives"]
        ),
        "false_positives": int(
            selected_calibrated_row["False Positives"]
        ),
        "false_negatives": int(
            selected_calibrated_row["False Negatives"]
        ),
        "true_negatives": int(
            selected_calibrated_row["True Negatives"]
        ),
        "flagged_skus": int(
            selected_calibrated_row["Flagged SKUs"]
        )
    },

    "holdout_evaluated": False,

    "prediction_time_availability_verified": False,

    "limitations": [
        "Calibration was evaluated on development data.",
        "The threshold was selected using development data.",
        "Business cost weights are hypothetical.",
        "Prediction-time feature availability is unverified.",
        "Temporal forecasting validity is not established.",
        "Final holdout evaluation is pending."
    ],

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

print("Calibration decision documentation: PASS")


In [ ]:

# Export calibration decision record

decision_file = (
    output_dir / "calibrated_model_decision.json"
)

with open(decision_file, "w", encoding="utf-8") as f:
    json.dump(
        calibration_decision,
        f,
        indent=4
    )

assert decision_file.exists()

print("Calibrated model decision export: PASS")
print("Saved to:", decision_file)


In [ ]:

# Phase 5 - Action 5.2.6D
# Verify calibration documentation

import numpy as np
import json

with open(decision_file, "r", encoding="utf-8") as f:
    saved_decision = json.load(f)

assert saved_decision["model"] == "Random Forest"

assert saved_decision["feature_count"] == 15

assert saved_decision["calibration_method"] == "sigmoid"

assert np.isclose(
    saved_decision["selected_threshold"],
    0.25
)

assert saved_decision["threshold_status"] == "provisional"

assert saved_decision["holdout_evaluated"] is False

assert saved_decision[
    "prediction_time_availability_verified"
] is False

assert calibration_comparison.shape == (5, 4)
assert calibrated_threshold_analysis.shape[0] == 18
assert best_calibrated_thresholds.shape[0] == 3

for filename in export_tables:
    assert (output_dir / filename).exists()

assert decision_file.exists()

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout.shape == (3049, 18)
assert X_unlabelled.shape == (3810, 18)

print("Calibration documentation integrity: PASS")
print("Calibrated threshold documentation: PASS")
print("All five exports verified: PASS")
print("Holdout status preserved: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.3.1
# Prepare final holdout feature matrix

import pandas as pd
import numpy as np

# Use the exact feature order established during development
final_feature_columns = list(reduced_features)

X_holdout_final = X_holdout.loc[
    :, final_feature_columns
].copy()

print("FINAL HOLDOUT PREPARATION")
print("-" * 45)

print("Development matrix:", X_dev_reduced.shape)
print("Holdout matrix:", X_holdout_final.shape)
print("Holdout labels:", y_holdout.shape)

print("\nSelected predictor count:",
      len(final_feature_columns))

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout_final.shape == (3049, 15)
assert y_holdout.shape == (3049,)

assert list(X_holdout_final.columns) == list(
    X_dev_reduced.columns
)

print("\nHoldout feature preparation: PASS")


In [ ]:

# Verify selected features and holdout structure

excluded_features = {
    "sku",
    "dataset_source",
    "went_on_backorder",
    "forecast_3_month",
    "forecast_6_month",
    "forecast_9_month",
    "pieces_past_due",
    "local_bo_qty",
    "potential_issue",
    "deck_risk",
    "oe_constraint",
    "ppap_risk",
    "stop_auto_buy",
    "rev_stop"
}

assert len(final_feature_columns) == 15
assert len(set(final_feature_columns)) == 15

assert not excluded_features.intersection(
    final_feature_columns
)

assert list(X_holdout_final.columns) == (
    list(X_dev_reduced.columns)
)

assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_holdout_final.dtypes
)

print("Selected predictor integrity: PASS")
print("Excluded columns absent: PASS")
print("Holdout feature alignment: PASS")
print("Numeric predictor types: PASS")


In [ ]:

# Verify development and holdout separation

development_indices = set(X_dev_reduced.index)
holdout_indices = set(X_holdout_final.index)

overlap_count = len(
    development_indices.intersection(holdout_indices)
)

print("TRAIN/HOLDOUT SEPARATION")
print("-" * 45)

print("Development records:", len(development_indices))
print("Holdout records:", len(holdout_indices))
print("Overlapping indices:", overlap_count)

assert len(development_indices) == 12194
assert len(holdout_indices) == 3049
assert overlap_count == 0

assert X_dev_reduced.index.equals(y_dev.index)
assert X_holdout_final.index.equals(y_holdout.index)

print("\nTrain/holdout separation: PASS")
print("Feature-label alignment: PASS")


In [ ]:

# Verify selected calibrated model configuration

from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

assert isinstance(
    calibrated_rf_model,
    CalibratedClassifierCV
)

assert calibrated_rf_model.method == "sigmoid"
assert calibrated_rf_model.ensemble is False

base_pipeline = calibrated_rf_model.estimator

assert isinstance(base_pipeline, Pipeline)

classifier = base_pipeline.named_steps["classifier"]

assert isinstance(classifier, RandomForestClassifier)

assert classifier.n_estimators == 200
assert classifier.max_depth == 12
assert classifier.min_samples_leaf == 5
assert classifier.class_weight == "balanced"
assert classifier.random_state == 42

assert np.isclose(
    provisional_calibrated_threshold,
    0.25
)

print("Calibrated model configuration: PASS")
print("Original Random Forest parameters: PASS")
print("Provisional threshold alignment: PASS")


In [ ]:

# Final holdout readiness verification

from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

# Create a fresh, unfitted copy for final evaluation
final_holdout_model = clone(calibrated_rf_model)

try:
    check_is_fitted(final_holdout_model)
    raise AssertionError(
        "The holdout model should not be fitted yet."
    )
except NotFittedError:
    print("Final model remains unfitted: PASS")

assert len(X_holdout_final) == len(y_holdout)

assert X_holdout_final.columns.equals(
    X_dev_reduced.columns
)

assert set(pd.Series(y_holdout).unique()).issubset(
    {0, 1}
)

assert X_unlabelled.shape == (3810, 18)

print("Holdout matrix readiness: PASS")
print("Binary target integrity: PASS")
print("Unlabelled dataset preserved: PASS")
print("\nFinal holdout preparation: PASS")


In [ ]:

# Phase 5 - Action 5.3.2
# Fit the selected model using development data only

from sklearn.utils.validation import check_is_fitted

final_holdout_model.fit(
    X_dev_reduced,
    y_dev
)

check_is_fitted(final_holdout_model)

print("Final calibrated model fitted: PASS")
print("Training records:", len(X_dev_reduced))
print("Training predictors:", X_dev_reduced.shape[1])
print("Holdout records used for fitting: 0")


In [ ]:

# Generate final holdout predictions

import numpy as np

holdout_probabilities = final_holdout_model.predict_proba(
    X_holdout_final
)[:, 1]

holdout_predictions = (
    holdout_probabilities >= provisional_calibrated_threshold
).astype(int)

assert len(holdout_probabilities) == 3049
assert len(holdout_predictions) == 3049

assert np.isfinite(holdout_probabilities).all()

assert (
    (holdout_probabilities >= 0)
    & (holdout_probabilities <= 1)
).all()

print("Holdout probability generation: PASS")
print("Holdout classification generation: PASS")

print("\nHoldout records:", len(holdout_predictions))
print("Selected threshold:", provisional_calibrated_threshold)
print("Flagged SKUs:", int(holdout_predictions.sum()))


In [ ]:

# Evaluate performance on the untouched holdout set

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

holdout_actual = y_holdout.to_numpy()

holdout_metrics = {
    "Average Precision": average_precision_score(
        holdout_actual, holdout_probabilities
    ),
    "ROC-AUC": roc_auc_score(
        holdout_actual, holdout_probabilities
    ),
    "Precision": precision_score(
        holdout_actual, holdout_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        holdout_actual, holdout_predictions,
        zero_division=0
    ),
    "F1-score": f1_score(
        holdout_actual, holdout_predictions,
        zero_division=0
    ),
    "Brier Score": brier_score_loss(
        holdout_actual, holdout_probabilities
    ),
    "Log Loss": log_loss(
        holdout_actual, holdout_probabilities,
        labels=[0, 1]
    )
}

print("FINAL HOLDOUT PERFORMANCE")
print("-" * 45)

for metric, value in holdout_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nFinal holdout metrics calculated: PASS")


In [ ]:

# Final holdout confusion matrix

tn, fp, fn, tp = confusion_matrix(
    holdout_actual,
    holdout_predictions,
    labels=[0, 1]
).ravel()

print("FINAL HOLDOUT CONFUSION MATRIX")
print("-" * 45)

print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)

print("\nActual backorders:", tp + fn)
print("Flagged SKUs:", tp + fp)
print("Holdout records:", tp + fp + fn + tn)

print("\nHoldout confusion matrix: PASS")


In [ ]:

# Validate final holdout evaluation

assert len(holdout_actual) == 3049
assert len(holdout_predictions) == 3049

assert set(np.unique(holdout_actual)).issubset({0, 1})
assert set(np.unique(holdout_predictions)).issubset({0, 1})

assert tp + fp + fn + tn == 3049

assert np.isfinite(
    list(holdout_metrics.values())
).all()

assert 0 <= holdout_metrics["Average Precision"] <= 1
assert 0 <= holdout_metrics["ROC-AUC"] <= 1
assert 0 <= holdout_metrics["Brier Score"] <= 1

assert np.isclose(
    provisional_calibrated_threshold,
    0.25
)

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout_final.shape == (3049, 15)

print("Holdout prediction integrity: PASS")
print("Holdout metric integrity: PASS")
print("Confusion matrix integrity: PASS")
print("Selected threshold preserved: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.3.3
# Visualize final holdout confusion matrix

import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

holdout_cm = confusion_matrix(
    holdout_actual,
    holdout_predictions,
    labels=[0, 1]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=holdout_cm,
    display_labels=["No Backorder", "Backorder"]
)

fig, ax = plt.subplots(figsize=(7, 6))

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title(
    "Final Holdout - Backorder Confusion Matrix"
)

plt.tight_layout()
plt.show()

assert holdout_cm.tolist() == [
    [2496, 190],
    [83, 280]
]

print("Holdout confusion matrix visualization: PASS")


In [ ]:

# Create holdout error-analysis dataset

import numpy as np
import pandas as pd

holdout_error_df = X_holdout_final.copy()

holdout_error_df["Actual_Backorder"] = holdout_actual
holdout_error_df["Predicted_Backorder"] = (
    holdout_predictions
)
holdout_error_df["Predicted_Probability"] = (
    holdout_probabilities
)

conditions = [
    (holdout_error_df["Actual_Backorder"] == 1)
    & (holdout_error_df["Predicted_Backorder"] == 1),

    (holdout_error_df["Actual_Backorder"] == 0)
    & (holdout_error_df["Predicted_Backorder"] == 1),

    (holdout_error_df["Actual_Backorder"] == 1)
    & (holdout_error_df["Predicted_Backorder"] == 0),

    (holdout_error_df["Actual_Backorder"] == 0)
    & (holdout_error_df["Predicted_Backorder"] == 0)
]

categories = [
    "True Positive",
    "False Positive",
    "False Negative",
    "True Negative"
]

holdout_error_df["Prediction_Outcome"] = np.select(
    conditions,
    categories,
    default="Unclassified"
)

print("HOLDOUT ERROR CLASSIFICATION")

display(
    holdout_error_df[
        [
            "Actual_Backorder",
            "Predicted_Backorder",
            "Predicted_Probability",
            "Prediction_Outcome",
            "national_inv",
            "lead_time"
        ]
    ].head(10)
)

print("\nHoldout error dataset created: PASS")


In [ ]:

# Compare inventory and supplier characteristics

error_profile = (
    holdout_error_df
    .groupby("Prediction_Outcome")
    .agg(
        SKU_Count=("Actual_Backorder", "size"),
        Median_Inventory=("national_inv", "median"),
        Median_In_Transit=("in_transit_qty", "median"),
        Median_Min_Bank=("min_bank", "median"),
        Median_Sales_3M=("sales_3_month", "median"),
        Median_Lead_Time=("lead_time", "median"),
        Mean_Predicted_Probability=(
            "Predicted_Probability", "mean"
        )
    )
    .reset_index()
)

print("HOLDOUT ERROR PROFILE")
display(error_profile.round(4))

print("\nHoldout error profiling: PASS")


In [ ]:

# Inspect false-negative backorder cases

missed_backorders = (
    holdout_error_df.loc[
        holdout_error_df["Prediction_Outcome"]
        == "False Negative"
    ]
    .sort_values(
        "Predicted_Probability",
        ascending=False
    )
)

print("MISSED BACKORDER ANALYSIS")
print("Total missed backorders:", len(missed_backorders))

display(
    missed_backorders[
        [
            "Predicted_Probability",
            "national_inv",
            "in_transit_qty",
            "min_bank",
            "sales_3_month",
            "lead_time",
            "perf_6_month_avg"
        ]
    ].head(15).round(4)
)

assert len(missed_backorders) == 83

print("\nMissed backorder inspection: PASS")


In [ ]:

# Inspect false-positive cases

false_alarms = (
    holdout_error_df.loc[
        holdout_error_df["Prediction_Outcome"]
        == "False Positive"
    ]
    .sort_values(
        "Predicted_Probability",
        ascending=False
    )
)

print("FALSE ALARM ANALYSIS")
print("Total false alarms:", len(false_alarms))

display(
    false_alarms[
        [
            "Predicted_Probability",
            "national_inv",
            "in_transit_qty",
            "min_bank",
            "sales_3_month",
            "lead_time",
            "perf_6_month_avg"
        ]
    ].head(15).round(4)
)

assert len(false_alarms) == 190

print("\nFalse alarm inspection: PASS")


In [ ]:

# Verify final holdout error analysis

expected_counts = {
    "True Positive": 280,
    "False Positive": 190,
    "False Negative": 83,
    "True Negative": 2496
}

actual_counts = (
    holdout_error_df["Prediction_Outcome"]
    .value_counts()
    .to_dict()
)

assert actual_counts == expected_counts

assert len(holdout_error_df) == 3049

assert (
    holdout_error_df["Prediction_Outcome"]
    != "Unclassified"
).all()

assert error_profile["SKU_Count"].sum() == 3049

assert len(missed_backorders) == 83
assert len(false_alarms) == 190

assert np.isclose(
    provisional_calibrated_threshold,
    0.25
)

assert X_holdout_final.shape == (3049, 15)

print("Prediction outcome classification: PASS")
print("Error group counts: PASS")
print("Inventory profile integrity: PASS")
print("Holdout threshold preserved: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.3.4
# Export final holdout evaluation metrics

import pandas as pd
from pathlib import Path

output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

holdout_metrics_df = pd.DataFrame([
    {
        "Metric": metric,
        "Value": float(value)
    }
    for metric, value in holdout_metrics.items()
])

metrics_file = (
    output_dir / "final_holdout_metrics.csv"
)

holdout_metrics_df.to_csv(
    metrics_file,
    index=False
)

print("FINAL HOLDOUT METRICS")
display(holdout_metrics_df.round(4))

assert metrics_file.exists()
assert holdout_metrics_df.shape == (7, 2)

print("\nHoldout metrics export: PASS")


In [ ]:

# Export final holdout confusion matrix

confusion_matrix_df = pd.DataFrame(
    holdout_cm,
    index=[
        "Actual_No_Backorder",
        "Actual_Backorder"
    ],
    columns=[
        "Predicted_No_Backorder",
        "Predicted_Backorder"
    ]
)

confusion_file = (
    output_dir / "final_holdout_confusion_matrix.csv"
)

confusion_matrix_df.to_csv(confusion_file)

print("FINAL HOLDOUT CONFUSION MATRIX")
display(confusion_matrix_df)

assert confusion_matrix_df.to_numpy().tolist() == [
    [2496, 190],
    [83, 280]
]

print("\nHoldout confusion matrix export: PASS")


In [ ]:

# Export holdout error profile

error_profile_file = (
    output_dir / "final_holdout_error_profile.csv"
)

error_profile.to_csv(
    error_profile_file,
    index=False
)

assert error_profile_file.exists()

assert error_profile["SKU_Count"].sum() == 3049

print("HOLDOUT ERROR PROFILE")
display(error_profile.round(4))

print("\nHoldout error profile export: PASS")


In [ ]:

# Document final holdout evaluation and limitations

import json
from datetime import datetime, timezone

final_evaluation_documentation = {
    "model": "Sigmoid-Calibrated Random Forest",
    "model_status": "holdout_evaluated",
    "feature_count": 15,
    "training_records": 12194,
    "holdout_records": 3049,
    "classification_threshold": float(
        provisional_calibrated_threshold
    ),
    "threshold_status": "provisional",
    "evaluation_design": (
        "Random stratified 80/20 development-holdout split"
    ),
    "holdout_metrics": {
        metric: float(value)
        for metric, value in holdout_metrics.items()
    },
    "confusion_matrix": {
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn)
    },
    "holdout_used_for_model_selection": False,
    "prediction_time_feature_availability_verified": False,
    "limitations": [
        "Data does not establish a reliable prediction timestamp.",
        "Prediction-time feature availability is unverified.",
        "Random holdout results do not prove future forecasting performance.",
        "Illustrative business-cost weights are not actual company costs.",
        "The operating threshold remains provisional.",
        "The model may not generalize to other companies or periods.",
        "Missing predictor values are imputed within the pipeline."
    ],
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

evaluation_file = (
    output_dir / "final_model_evaluation.json"
)

with open(evaluation_file, "w", encoding="utf-8") as f:
    json.dump(
        final_evaluation_documentation,
        f,
        indent=4
    )

assert evaluation_file.exists()

print("Final evaluation documentation: PASS")
print("Saved:", evaluation_file.name)


In [ ]:

# Verify final model evaluation records

import numpy as np

with open(evaluation_file, "r", encoding="utf-8") as f:
    saved_evaluation = json.load(f)

assert saved_evaluation["holdout_records"] == 3049

assert saved_evaluation["feature_count"] == 15

assert np.isclose(
    saved_evaluation["classification_threshold"],
    0.25
)

assert saved_evaluation[
    "holdout_used_for_model_selection"
] is False

assert saved_evaluation[
    "prediction_time_feature_availability_verified"
] is False

assert len(saved_evaluation["holdout_metrics"]) == 7

assert sum(
    saved_evaluation["confusion_matrix"].values()
) == 3049

assert all(
    path.exists()
    for path in [
        metrics_file,
        confusion_file,
        error_profile_file,
        evaluation_file
    ]
)

assert X_dev_reduced.shape == (12194, 15)
assert X_holdout_final.shape == (3049, 15)

print("Final holdout metrics integrity: PASS")
print("Confusion matrix documentation: PASS")
print("Error profile documentation: PASS")
print("Holdout independence documented: PASS")
print("All four evaluation exports: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.4.1
# Save the fitted calibrated Random Forest

import joblib
from pathlib import Path
from sklearn.utils.validation import check_is_fitted

# Ensure the model is already trained
check_is_fitted(final_holdout_model)

output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

model_file = (
    output_dir / "calibrated_random_forest.joblib"
)

joblib.dump(
    final_holdout_model,
    model_file
)

assert model_file.exists()
assert model_file.stat().st_size > 0

print("Trained model export: PASS")
print("Model saved to:", model_file)
print(
    "File size (MB):",
    round(model_file.stat().st_size / (1024 ** 2), 2)
)


In [ ]:

# Save feature schema and model configuration

import json
import sklearn
from datetime import datetime, timezone

model_metadata = {
    "model_name": "Sigmoid-Calibrated Random Forest",
    "model_file": "calibrated_random_forest.joblib",
    "model_stage": "historical_holdout_validated",
    "feature_count": len(final_feature_columns),
    "feature_columns": list(final_feature_columns),

    "calibration_method": "sigmoid",
    "classification_threshold": float(
        provisional_calibrated_threshold
    ),
    "threshold_status": "provisional",

    "training_records": 12194,
    "holdout_records": 3049,
    "holdout_used_for_training": False,

    "prediction_time_feature_availability_verified": False,
    "temporal_forecasting_validated": False,

    "python_sklearn_version": sklearn.__version__,
    "saved_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

metadata_file = (
    output_dir / "calibrated_random_forest_metadata.json"
)

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=4)

assert metadata_file.exists()
assert len(model_metadata["feature_columns"]) == 15

print("Model metadata export: PASS")
print("Feature count:", model_metadata["feature_count"])
print("Threshold:", model_metadata["classification_threshold"])


In [ ]:

# Reload the saved model

loaded_model = joblib.load(model_file)

check_is_fitted(loaded_model)

print("Saved model reloaded: PASS")
print("Model type:", type(loaded_model).__name__)

assert loaded_model.method == "sigmoid"

print("Calibration configuration preserved: PASS")


In [ ]:

# Verify saved model prediction consistency

import numpy as np

reloaded_probabilities = loaded_model.predict_proba(
    X_holdout_final
)[:, 1]

max_probability_difference = np.max(
    np.abs(
        reloaded_probabilities - holdout_probabilities
    )
)

assert np.allclose(
    reloaded_probabilities,
    holdout_probabilities,
    rtol=0,
    atol=1e-12
)

reloaded_predictions = (
    reloaded_probabilities
    >= provisional_calibrated_threshold
).astype(int)

assert np.array_equal(
    reloaded_predictions,
    holdout_predictions
)

print("Saved model probability consistency: PASS")
print("Saved model classification consistency: PASS")

print(
    "Maximum probability difference:",
    max_probability_difference
)


In [ ]:

# Final saved model validation

with open(metadata_file, "r", encoding="utf-8") as f:
    saved_metadata = json.load(f)

assert saved_metadata["feature_count"] == 15

assert saved_metadata["feature_columns"] == (
    list(X_dev_reduced.columns)
)

assert saved_metadata["feature_columns"] == (
    list(X_holdout_final.columns)
)

assert np.isclose(
    saved_metadata["classification_threshold"],
    0.25
)

assert saved_metadata["calibration_method"] == "sigmoid"

assert saved_metadata["holdout_used_for_training"] is False

assert saved_metadata[
    "prediction_time_feature_availability_verified"
] is False

assert model_file.exists()
assert metadata_file.exists()

assert X_unlabelled.shape == (3810, 18)

print("Feature schema preservation: PASS")
print("Threshold metadata integrity: PASS")
print("Calibration metadata integrity: PASS")
print("Saved model files verification: PASS")
print("Unlabelled dataset preserved: PASS")


In [ ]:

# Phase 5 - Action 5.4.2
# Prepare unlabelled SKUs for risk scoring

import pandas as pd
import numpy as np

scoring_features = saved_metadata["feature_columns"]

X_unlabelled_final = X_unlabelled.loc[
    :, scoring_features
].copy()

print("UNLABELLED SKU SCORING PREPARATION")
print("-" * 45)

print("Original matrix:", X_unlabelled.shape)
print("Scoring matrix:", X_unlabelled_final.shape)
print("SKU identifiers:", len(unlabelled_sku_ids))
print("Selected predictors:", len(scoring_features))

assert X_unlabelled.shape == (3810, 18)
assert X_unlabelled_final.shape == (3810, 15)
assert len(unlabelled_sku_ids) == 3810

assert list(X_unlabelled_final.columns) == (
    saved_metadata["feature_columns"]
)

assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_unlabelled_final.dtypes
)

print("\nUnlabelled feature preparation: PASS")


In [ ]:

# Generate calibrated backorder risk scores

sku_risk_probabilities = loaded_model.predict_proba(
    X_unlabelled_final
)[:, 1]

print("SKU RISK SCORING")
print("-" * 45)

print("SKUs scored:", len(sku_risk_probabilities))

print(
    "Minimum risk score:",
    round(float(sku_risk_probabilities.min()), 4)
)

print(
    "Maximum risk score:",
    round(float(sku_risk_probabilities.max()), 4)
)

print(
    "Mean risk score:",
    round(float(sku_risk_probabilities.mean()), 4)
)

assert len(sku_risk_probabilities) == 3810
assert np.isfinite(sku_risk_probabilities).all()

assert (
    (sku_risk_probabilities >= 0)
    & (sku_risk_probabilities <= 1)
).all()

print("\nCalibrated SKU risk scoring: PASS")


In [ ]:

# Apply the provisional calibrated classification threshold

scoring_threshold = float(
    saved_metadata["classification_threshold"]
)

assert np.isclose(scoring_threshold, 0.25)

sku_risk_predictions = (
    sku_risk_probabilities >= scoring_threshold
).astype(int)

print("SKU RISK CLASSIFICATION")
print("-" * 45)

print("Classification threshold:", scoring_threshold)
print("SKUs flagged:", int(sku_risk_predictions.sum()))

print(
    "SKUs not flagged:",
    int((sku_risk_predictions == 0).sum())
)

print(
    "Flagged percentage:",
    f"{sku_risk_predictions.mean():.2%}"
)

assert len(sku_risk_predictions) == 3810

assert set(np.unique(sku_risk_predictions)).issubset(
    {0, 1}
)

print("\nSKU risk classification: PASS")


In [ ]:

# Create SKU-level risk prediction table

sku_ids = pd.Series(
    unlabelled_sku_ids.to_numpy()
).reset_index(drop=True)

sku_risk_results = pd.DataFrame({
    "sku": sku_ids,
    "predicted_backorder_probability":
        sku_risk_probabilities,
    "predicted_backorder_flag":
        sku_risk_predictions,
    "classification_threshold":
        scoring_threshold,
    "model_name":
        "Sigmoid-Calibrated Random Forest"
})

print("SKU-LEVEL RISK PREDICTIONS")

display(
    sku_risk_results.head(10).round(4)
)

print("\nSKU-level prediction table: PASS")


In [ ]:

# Verify SKU-level prediction integrity

assert sku_risk_results.shape == (3810, 5)

assert sku_risk_results["sku"].notna().all()

assert sku_risk_results["sku"].nunique() == 3810

assert not sku_risk_results["sku"].duplicated().any()

assert (
    sku_risk_results[
        "predicted_backorder_probability"
    ].between(0, 1)
).all()

assert sku_risk_results[
    "predicted_backorder_probability"
].notna().all()

assert (
    sku_risk_results[
        "predicted_backorder_flag"
    ].to_numpy()
    == sku_risk_predictions
).all()

assert (
    sku_risk_results["predicted_backorder_flag"].sum()
    + (sku_risk_results["predicted_backorder_flag"] == 0).sum()
    == 3810
)

assert X_holdout_final.shape == (3049, 15)

print("SKU identifier integrity: PASS")
print("Predicted probability integrity: PASS")
print("Classification threshold consistency: PASS")
print("SKU prediction table integrity: PASS")
print("Holdout dataset preserved: PASS")


In [ ]:

# Phase 5 - Action 5.4.3
# Create provisional SKU risk categories

import numpy as np
import pandas as pd

risk_threshold = float(
    provisional_calibrated_threshold
)

assert np.isclose(risk_threshold, 0.25)

# Preserve the original prediction results
sku_risk_categorized = sku_risk_results.copy()

risk_scores = sku_risk_categorized[
    "predicted_backorder_probability"
]

sku_risk_categorized["risk_category"] = np.select(
    [
        risk_scores < 0.10,
        (risk_scores >= 0.10) & (risk_scores < risk_threshold),
        risk_scores >= risk_threshold
    ],
    [
        "Low",
        "Medium",
        "High"
    ],
    default="Unclassified"
)

print("SKU RISK CATEGORIES")
display(sku_risk_categorized.head(10).round(4))

print("\nRisk categories created: PASS")


In [ ]:

# Calculate SKU risk category distribution

risk_category_order = ["Low", "Medium", "High"]

risk_distribution = (
    sku_risk_categorized["risk_category"]
    .value_counts()
    .reindex(risk_category_order, fill_value=0)
    .rename_axis("Risk Category")
    .reset_index(name="SKU Count")
)

risk_distribution["Percentage"] = (
    risk_distribution["SKU Count"]
    / len(sku_risk_categorized)
    * 100
)

print("RISK CATEGORY DISTRIBUTION")
display(risk_distribution.round(2))

print("\nTotal SKUs:", risk_distribution["SKU Count"].sum())

print("\nRisk category distribution: PASS")


In [ ]:

# Visualize SKU risk categories

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    risk_distribution["Risk Category"],
    risk_distribution["SKU Count"]
)

for bar in bars:
    height = bar.get_height()
    ax.annotate(
        f"{int(height):,}",
        (bar.get_x() + bar.get_width() / 2, height),
        ha="center",
        va="bottom",
        xytext=(0, 4),
        textcoords="offset points"
    )

ax.set_title("Unlabelled SKUs by Model-Estimated Risk")
ax.set_xlabel("Risk Category")
ax.set_ylabel("Number of SKUs")
ax.set_ylim(
    0,
    max(risk_distribution["SKU Count"]) * 1.15
)

plt.tight_layout()
plt.show()

print("Risk distribution chart: PASS")


In [ ]:

# Audit model score distribution by risk category

risk_score_audit = (
    sku_risk_categorized
    .groupby("risk_category")
    .agg(
        SKU_Count=("sku", "size"),
        Minimum_Score=(
            "predicted_backorder_probability", "min"
        ),
        Median_Score=(
            "predicted_backorder_probability", "median"
        ),
        Mean_Score=(
            "predicted_backorder_probability", "mean"
        ),
        Maximum_Score=(
            "predicted_backorder_probability", "max"
        ),
        Flagged_SKUs=(
            "predicted_backorder_flag", "sum"
        )
    )
    .reindex(risk_category_order)
    .reset_index()
)

print("RISK SCORE AUDIT")
display(risk_score_audit.round(4))

print("\nRisk score audit: PASS")


In [ ]:

# Inspect highest-scoring unlabelled SKUs

top_risk_skus = (
    sku_risk_categorized
    .sort_values(
        "predicted_backorder_probability",
        ascending=False
    )
    .head(15)
)

print("TOP 15 MODEL-ESTIMATED RISK SKUS")

display(
    top_risk_skus[
        [
            "sku",
            "predicted_backorder_probability",
            "predicted_backorder_flag",
            "risk_category"
        ]
    ].round(4)
)

print("\nHigh-risk SKU inspection: PASS")


In [ ]:

# Final risk category validation

assert sku_risk_categorized.shape == (3810, 6)

assert sku_risk_categorized["sku"].nunique() == 3810

assert sku_risk_categorized["risk_category"].isin(
    ["Low", "Medium", "High"]
).all()

assert risk_distribution["SKU Count"].sum() == 3810

assert np.isclose(
    risk_distribution["Percentage"].sum(),
    100.0
)

assert (
    sku_risk_categorized.loc[
        sku_risk_categorized["risk_category"] == "Low",
        "predicted_backorder_probability"
    ] < 0.10
).all()

assert (
    sku_risk_categorized.loc[
        sku_risk_categorized["risk_category"] == "Medium",
        "predicted_backorder_probability"
    ].between(0.10, risk_threshold, inclusive="left")
).all()

assert (
    sku_risk_categorized.loc[
        sku_risk_categorized["risk_category"] == "High",
        "predicted_backorder_probability"
    ] >= risk_threshold
).all()

assert (
    sku_risk_categorized["risk_category"].eq("High").to_numpy()
    == sku_risk_categorized[
        "predicted_backorder_flag"
    ].eq(1).to_numpy()
).all()

assert (
    sku_risk_categorized["predicted_backorder_flag"].sum()
    == 641
)

assert X_unlabelled_final.shape == (3810, 15)
assert X_holdout_final.shape == (3049, 15)

print("Risk category completeness: PASS")
print("SKU identifier uniqueness: PASS")
print("Risk score boundary integrity: PASS")
print("High-risk classification alignment: PASS")
print("641 flagged SKUs preserved: PASS")
print("Dataset dimensions preserved: PASS")


In [ ]:

# Phase 5 - Action 5.4.4
# Prepare SKU-level risk prediction export

import pandas as pd
import numpy as np

export_columns = [
    "sku",
    "predicted_backorder_probability",
    "predicted_backorder_flag",
    "classification_threshold",
    "model_name",
    "risk_category"
]

sku_risk_export = sku_risk_categorized[
    export_columns
].copy()

print("FINAL SKU RISK EXPORT PREPARATION")
print("-" * 45)

print("Dataset dimensions:", sku_risk_export.shape)
print("Unique SKUs:", sku_risk_export["sku"].nunique())
print("Flagged SKUs:",
      int(sku_risk_export["predicted_backorder_flag"].sum()))

print("\nRisk categories:")
print(sku_risk_export["risk_category"].value_counts())

assert sku_risk_export.shape == (3810, 6)
assert sku_risk_export["sku"].nunique() == 3810
assert sku_risk_export["predicted_backorder_flag"].sum() == 641

print("\nSKU risk export preparation: PASS")


In [ ]:

# Export predictions for Power BI integration

from pathlib import Path

output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

sku_prediction_file = (
    output_dir / "sku_backorder_risk_predictions.csv"
)

sku_risk_export.to_csv(
    sku_prediction_file,
    index=False,
    float_format="%.10f"
)

assert sku_prediction_file.exists()
assert sku_prediction_file.stat().st_size > 0

print("SKU prediction CSV export: PASS")
print("Saved to:", sku_prediction_file)
print("Exported records:", len(sku_risk_export))


In [ ]:

# Reload exported CSV for integrity validation

sku_risk_reloaded = pd.read_csv(
    sku_prediction_file,
    dtype={"sku": "int64"}
)

assert sku_risk_reloaded.shape == (3810, 6)

assert list(sku_risk_reloaded.columns) == export_columns

assert sku_risk_reloaded["sku"].nunique() == 3810

assert not sku_risk_reloaded["sku"].duplicated().any()

assert sku_risk_reloaded["sku"].notna().all()

assert np.allclose(
    sku_risk_reloaded["predicted_backorder_probability"],
    sku_risk_export["predicted_backorder_probability"],
    rtol=0,
    atol=1e-9
)

assert (
    sku_risk_reloaded["predicted_backorder_flag"]
    .to_numpy()
    == sku_risk_export["predicted_backorder_flag"]
    .to_numpy()
).all()

assert (
    sku_risk_reloaded["risk_category"]
    .to_numpy()
    == sku_risk_export["risk_category"]
    .to_numpy()
).all()

print("CSV reload and schema validation: PASS")
print("SKU uniqueness and integrity: PASS")
print("Probability preservation: PASS")
print("Classification preservation: PASS")
print("Risk category preservation: PASS")


In [ ]:

# Verify category distribution after CSV export

export_distribution = (
    sku_risk_reloaded["risk_category"]
    .value_counts()
    .reindex(["Low", "Medium", "High"], fill_value=0)
)

expected_distribution = pd.Series({
    "Low": 2947,
    "Medium": 222,
    "High": 641
})

assert export_distribution.equals(expected_distribution)

assert export_distribution.sum() == 3810

assert np.isclose(
    sku_risk_reloaded["classification_threshold"],
    0.25
).all()

assert (
    sku_risk_reloaded["predicted_backorder_flag"]
    .to_numpy()
    == (
        sku_risk_reloaded["predicted_backorder_probability"]
        >= 0.25
    ).astype(int).to_numpy()
).all()

assert (
    sku_risk_reloaded["risk_category"].eq("High")
    .to_numpy()
    == sku_risk_reloaded["predicted_backorder_flag"]
    .eq(1).to_numpy()
).all()

print("EXPORTED RISK CATEGORY DISTRIBUTION")
display(
    export_distribution.rename("SKU Count").to_frame()
)

print("\nExported risk distribution: PASS")
print("Threshold and flag consistency: PASS")


In [ ]:

# Verify exact original unlabelled SKU coverage

source_skus = set(
    pd.Series(unlabelled_sku_ids).astype("int64")
)

exported_skus = set(
    sku_risk_reloaded["sku"]
)

missing_skus = source_skus - exported_skus
unexpected_skus = exported_skus - source_skus

print("SOURCE SKU COVERAGE AUDIT")
print("-" * 45)

print("Source unlabelled SKUs:", len(source_skus))
print("Exported unique SKUs:", len(exported_skus))
print("Missing SKUs:", len(missing_skus))
print("Unexpected SKUs:", len(unexpected_skus))

assert len(source_skus) == 3810
assert len(exported_skus) == 3810

assert len(missing_skus) == 0
assert len(unexpected_skus) == 0

print("\nOriginal SKU coverage: PASS")


In [ ]:

# Final SKU prediction export sign-off

assert sku_prediction_file.exists()

assert sku_risk_reloaded.shape == (3810, 6)

assert sku_risk_reloaded["sku"].is_unique

assert (
    sku_risk_reloaded[
        "predicted_backorder_probability"
    ].between(0, 1)
).all()

assert sku_risk_reloaded[
    "predicted_backorder_probability"
].notna().all()

assert (
    sku_risk_reloaded["predicted_backorder_flag"].sum()
    == 641
)

assert (
    sku_risk_reloaded["risk_category"]
    .value_counts()
    .to_dict()
    == {"Low": 2947, "Medium": 222, "High": 641}
)

assert X_holdout_final.shape == (3049, 15)
assert X_unlabelled_final.shape == (3810, 15)

print("Final CSV existence: PASS")
print("SKU-level export integrity: PASS")
print("Risk score validity: PASS")
print("641 flagged SKUs preserved: PASS")
print("Three risk categories preserved: PASS")
print("Model dataset dimensions preserved: PASS")

print("\nAction 5.4.4 export verification: PASS")
